# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset, *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*, using the `mlcroissant` library. The dataset explores protein abundance, modifications, and sequences measured via mass spectrometry from extracellular vesicles of human mast cells.

### Dataset Source
The dataset is published as a FAIR-compliant Croissant schema and accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant dataset metadata and records using `mlcroissant`. This will retrieve and parse all record sets, fields, and definitions specified in the schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata, e.g., name and description
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

# (Optional) Print publication date and data use cases if available
if hasattr(dataset.metadata, 'datePublished'):
    print(f"Published: {dataset.metadata.datePublished}")
if hasattr(dataset.metadata, 'dataUseCases'):
    print(f"Data Use Cases: {dataset.metadata.dataUseCases}")

## 2. Data Overview
Enumerate available record sets, their `@id` values, and their fields. The `@id` will be used for all further data access in alignment with the Croissant data model and the notebook template.

In [ ]:
# List all record sets and their fields using @id references
print("All Record Sets in the Dataset:")
recordsets = list(dataset.record_sets)
for recset in recordsets:
    print(f"- RecordSet name: {recset.name}")
    print(f"  @id: {recset.id}")
    print(f"  Fields:")
    for field in recset.fields:
        print(f"    - Field name: {field.name}   @id: {field.id}   dataType: {getattr(field, 'data_type', 'N/A')}")
    print()
print(f"Total Record Sets: {len(recordsets)}")

To view the first few records for a given record set by `@id` (from output above), call `dataset.records(record_set='<record_set_id>')`. For example, to print the top 3 records of the first record set:

In [ ]:
# Preview records for a specific RecordSet using its @id
# Replace with the actual @id you wish to explore, e.g.:
# record_set_id = '<your_record_set_id>'
if len(recordsets) > 0:
    record_set_id = recordsets[0].id  # We take the first record set by default
    print(f"Top 3 records for RecordSet @id={record_set_id}:\n")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Extract data from one or more record sets into pandas DataFrames using their `@id`.

Review the previously listed field `@id`s and select those relevant to your analysis.

In [ ]:
# Example: Load all record sets into pandas DataFrames
dataframes = {}
record_set_ids = [recset.id for recset in recordsets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded @{record_set_id} with columns: {df.columns.tolist()}")
        print(df.head(2))
        print()
    else:
        print(f"RecordSet @{record_set_id} is empty or not tabular.")

# For analysis, pick the main protein abundance record set (example name matching)
main_record_set_id = None
for recset in recordsets:
    lowername = recset.name.lower() if recset.name else ''
    if 'protein' in lowername or 'abundance' in lowername or 'vesicle' in lowername:
        main_record_set_id = recset.id
        break
if not main_record_set_id and len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"Using default RecordSet: {main_record_set_id}")

# Show DataFrame columns for main_record_set_id
if main_record_set_id in dataframes:
    print(f"Columns in main analysis DataFrame (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Process and explore the records: filter, normalize, and summarize numeric fields (e.g., protein abundance, peptide count, coverage percentage, molecular weight, etc.).

Below, replace `<numeric_field_id>` and `<group_field>` with appropriate column names or field @ids from the DataFrame.

In [ ]:
# Example: EDA on a numeric column
df = dataframes[main_record_set_id]
print(f"Available columns: {df.columns.tolist()}")

# Try using common numeric fields
numeric_field_candidates = ['abundance', 'Abundance', 'MW', 'Molecular Weight', 'peptide_count', 'Coverage']
numeric_field = None
for col in df.columns:
    for candidate in numeric_field_candidates:
        if candidate.lower() in col.lower():
            numeric_field = col
            break
    if numeric_field:
        break
if not numeric_field:
    # Fallback: use the first numeric column detected
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
print(f"Selected numeric field for analysis: {numeric_field}")

# Remove obvious outliers (threshold is arbitrary for illustration)
threshold = df[numeric_field].mean() + 3 * df[numeric_field].std() if numeric_field else None
if numeric_field and threshold:
    filtered_df = df[df[numeric_field] < threshold]
    print(f"Filtered records with {numeric_field} < {threshold:.2f} (approx 3-sigma):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Identify a grouping field
    group_field = None
    group_candidates = ['Sample', 'sample', 'Replicate', 'Condition', 'Group']
    for col in df.columns:
        for gc in group_candidates:
            if gc.lower() in col.lower():
                group_field = col
                break
        if group_field:
            break
    print(f"Group field for aggregation: {group_field}")
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("Could not identify a numeric field suitable for analysis.")

## 5. Visualization
Visualize a histogram of the selected numeric field and, if available, the distribution by a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- Loaded dataset metadata and displayed core information.
- Explored available record sets, fields, their `@id`, and tabular content.
- Filtered, normalized, and aggregated key numeric fields (e.g., protein abundances) using Croissant's explicit `@id` referencing.
- Visualized distributions and group-wise statistics.

This notebook can serve as a template for further, deeper analyses as needed—modifying record set and field IDs as appropriate for your research questions.